# mf6adj demonstration using the synthetic dewatering example

In this notebook, we show how mf6adj can be used with a MODFLOW 6 version of the synthetic mine dewatering example presented by White et al. (2025), "Reliable Trade-offs Between Environment and Economy: Implications for Mine Dewatering and Managed Aquifer Recharge."

## Import Packages

In [ ]:
import os
import pathlib as pl
import platform
import shutil
import sys
from datetime import datetime

import flopy
import h5py
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import pyemu

In [ ]:
try:
    import mf6adj
except ImportError:
    sys.path.insert(0, str(pl.Path("../").resolve()))
    import mf6adj

In [ ]:
mf6_bin, lib_name = mf6adj.get_conda_mf6_paths()
print(f"Using MF6 binary: {mf6_bin}")
print(f"Using MF6 library: {lib_name}")

Now let's get the model files we will use. They are stored in the `autotest` directory.

## Copy the MODFLOW Model

In [ ]:
org_ws = pl.Path("synthdewater")
assert pl.Path(org_ws).exists()

Set up a local copy of the model files.

In [ ]:
ws = pl.Path("synthdewater_working")
if pl.Path(ws).exists():
    shutil.rmtree(ws)
shutil.copytree(org_ws, ws)

In [ ]:
sim = flopy.mf6.MFSimulation.load(sim_ws=ws)
m = sim.get_model()
X, Y = m.modelgrid.xcellcenters, m.modelgrid.ycellcenters

In [ ]:
def plot_model(k, arr, units=None, cmap="plasma", center=False, levels=None):
    vmin = None
    vmax = None

    if center:
        mx = np.nanmax(np.abs(arr))
        vmin = -1.0 * mx
        vmax = mx
        cmap = "coolwarm"

    fig, ax = plt.subplots(1, 1, figsize=(6, 6))
    ax.set_aspect("equal")
    mv = flopy.plot.PlotMapView(model=m, ax=ax)
    mv.plot_bc("WEL-dewater", label="Dewater Wells")
    mv.plot_bc("WEL-mar", label="Injection Wells")

    mv.plot_bc("DRN", color="green", label="Drain - GDE")
    mv.plot_bc("GHB", color="blue", label="GHB - regional aquifer")
    cb = ax.pcolormesh(X, Y, arr, cmap=cmap, vmin=vmin, vmax=vmax, alpha=0.5)

    plt.colorbar(cb, ax=ax, label=units)
    if levels is not None:
        c = "w"
        if center:
            c = "k"
        CS = ax.contour(X, Y, arr, levels=levels, colors=c)
        ax.clabel(CS, CS.levels, fontsize=10)

    return fig, ax

In [ ]:
fig, ax = plot_model(0, m.dis.idomain.array[0, :, :].astype(float))
_ = ax.set_title("idomain")

The pit is located in the `idomain == 2` region.

In [ ]:
fig, ax = plt.subplots()
mv = flopy.plot.PlotMapView(model=m)
mv.plot_grid(lw=0.5)
mv.plot_bc("WEL-dewater", label="Dewater Wells")
mv.plot_bc("WEL-mar", label="Injection Wells")

mv.plot_bc("DRN", color="green", label="Drain - GDE")
mv.plot_bc("GHB", color="blue", label="GHB - regional aquifer")

# mv.plot_array(gwf.dis.idomain.get_data(), color='gray', alpha=1)

Here we see the GDE DRN boundary on the left, the inflow GHB on the right, and the dewatering and reinjection wells. The model has three stress periods: pre-development (steady state), active mining (10-year transient), and closure (20-year transient).

In [ ]:
fig, ax = plot_model(0, np.log10(m.npf.k.array[0, :, :]), levels=3)
_ = ax.set_title("HK")

A beautiful example of nonstationary geostatistics.

Run the existing model in our local workspace.

In [ ]:
pyemu.os_utils.run(mf6_bin.name, cwd=ws)

Now plot some heads.

In [ ]:
labels = ["predev", "end of mining", "closure"]
hds = flopy.utils.HeadFile(pl.Path(ws) / "model.hds")
for kper, label in enumerate(labels):
    final_arr = hds.get_data(kstpkper=(0, kper))
    fig, ax = plot_model(0, final_arr[0, :, :], units="meters", levels=5)
    ax.set_title(label)

As expected, groundwater flows from high head to low head.

The main requirement for using mf6adj is an input file that describes the performance measures. Fortunately, this file follows a modern format similar to other MF6 input files. Here we will build these measures programmatically so we can examine the pit head at the end of mining and the GDE-boundary flux during pre-development, end-of-mining, and post-closure conditions.

In [ ]:
drn = pd.DataFrame.from_records(m.drn.stress_period_data.array[0])
drn

In [ ]:
names = ["drn-gde-predev", "drn-gde-endmining", "drn-gde-postclosure"]
pm_fname = "prefmeas.dat"
fpm = open(pl.Path(ws) / pm_fname, "w")

for kper, name in enumerate(names):
    fpm.write(f"begin performance_measure {name}\n")

    for kij in drn.cellid.values:
        fpm.write(
            f"{kper + 1} 1 {kij[0] + 1} {kij[1] + 1} {kij[2] + 1} "
            + "drn-gde direct 1.0 -1.0e+30\n"
        )
    fpm.write("end performance_measure\n\n")

In [ ]:
name = "pithead-endmining"

kij = (0, 49, 49)
fpm.write(f"begin performance_measure {name}\n")
fpm.write(
    f"{kper + 1} 1 {kij[0] + 1} {kij[1] + 1} {kij[2] + 1} head direct 1.0 -1.0e+30\n"
)
fpm.write("end performance_measure\n\n")
fpm.close()

## Run mf6adj

Now we should be ready to go. The adjoint solution process requires one forward model run and then a solve for the adjoint state, which uses the forward solution components (for example, the conductance matrix, RHS, heads, and saturation). The adjoint solve has two important characteristics: it is linear, regardless of the forward model's linearity, and it proceeds backward in time, starting with the last stress period.

The adjoint solve is slower than the forward run because most of the time is spent in the NumPy sparse linear solve.

In [ ]:
forward_hdf5_name = "forward.hdf5"
start = datetime.now()

adj = mf6adj.Mf6Adj(
    pm_fname,
    lib_name,
    logging_level="INFO",
    working_directory=ws,
)
adj.solve_forward_model(
    hdf5_name=forward_hdf5_name
)  # solve the standard forward solution
dfsum = adj.solve_adjoint()  # solve the adjoint state for each performance measure
adj.finalize()  # release components
duration = (datetime.now() - start).total_seconds()
print("took:", duration)

## Plot the Results

Done. Let's see what happened.

In [ ]:
hdf5_files = [f for f in os.listdir(ws) if f.endswith("hdf5")]
hdf5_files.sort()
hdf5_files = hdf5_files[:-1]
hdf5_files

mf6adj uses the widely available HDF5 format to store information. These files hold low-level details about the adjoint solution. However, mf6adj.solve_adjoint() also returns a higher-level summary of the results. Let's look at that first:

In [ ]:
type(dfsum)

In [ ]:
list(dfsum.keys())

In [ ]:
dfhw = dfsum["pithead-endmining"]
dfhw

Those are the node-scale sensitivities for the end-of-mining pit groundwater-level performance measure. Plotting them is easiest using the HDF5 file itself.

In [ ]:
result_hdf = hdf5_files[-1]
hdf = h5py.File(pl.Path(ws) / result_hdf, "r")
keys = list(hdf.keys())
keys.sort()
print(keys)

The `composite` group contains sensitivities of the performance measure to the model inputs, summed across all adjoint solutions.

In [ ]:
grp = hdf["composite"]
plot_keys = [
    i for i in grp.keys() if len(grp[i].shape) == 3 and ("k11" in i or "ss" in i)
]
plot_keys

Here is a simple routine to plot these sensitivities.

In [ ]:
for pkey in plot_keys:
    arr = grp[pkey][:]
    for k, karr in enumerate(arr):
        karr[karr == 0.0] = np.nan
        fig, ax = plot_model(k, karr, center=True, levels=4)
        ax.set_title(pkey + f", layer:{k + 1}", loc="left")

These are the sensitivity maps for end-of-mining pit groundwater level with respect to `hk` and `ss`. Are these patterns what you expected?

Now let's look at the same plots for the GDE-flux performance measures:

In [ ]:
for result_hdf in hdf5_files:
    hdf = h5py.File(pl.Path(ws) / result_hdf, "r")
    grp = hdf["composite"]
    pm_name = (
        result_hdf.replace("adjoint_solution_", "")
        .split(".")[0]
        .replace("_forward", "")
    )
    for pkey in plot_keys:
        arr = grp[pkey][:]
        for k, karr in enumerate(arr):
            karr[karr == 0.0] = np.nan
            fig, ax = plot_model(k, karr, center=True, levels=4)
            ax.set_title(pm_name + ", " + pkey, loc="left")

Are those sensitivities consistent with your expectations and hydrologic intuition? Let's look at the `hk` array again:

In [ ]:
fig, ax = plot_model(0, np.log10(m.npf.k.array[0, :, :]), levels=3)
_ = ax.set_title("HK")

Let's also look at the sensitivity of groundwater pumping during the active mining period to the head in the pit at the end of mining:

In [ ]:
keys

In [ ]:
mining_grp = [key for key in keys if key.startswith("solution_kper:00001")]
assert len(mining_grp) == 1
mining_grp = mining_grp[0]
mining_grp

In [ ]:
plot_keys = [i for i in grp.keys() if len(grp[i].shape) == 3 and ("wel6_q" in i)]
plot_keys

In [ ]:
for result_hdf in hdf5_files:
    hdf = h5py.File(pl.Path(ws) / result_hdf, "r")
    grp = hdf[mining_grp]
    pm_name = (
        result_hdf.replace("adjoint_solution_", "")
        .split(".")[0]
        .replace("_forward", "")
    )
    for pkey in plot_keys:
        arr = grp[pkey][:]
        for k, karr in enumerate(arr):
            karr[karr == 0.0] = np.nan
            fig, ax = plot_model(k, karr, center=True, levels=4)
            ax.set_title(pm_name + ", " + pkey, loc="left")

This helps show how groundwater extraction and reinjection during active mining influence both pit groundwater levels and GDE flux at the end of mining and at the end of the closure period.